# pmap Notebook Test

Tests that `pmap` auto-detects notebook environment and uses the tqdm backend.
Also tests explicit backend override with `backend='tqdm'`.

In [ ]:
import time
from loguru import logger
import rich
from pmap import pmap, is_notebook

ITEMS = range(5)
EXPECTED = [0, 2, 4, 6, 8]

def worker(i: int) -> int:
    """Worker with all output methods."""
    print(f"print: item {i}")
    rich.print(f"[bold cyan]rich: item {i}[/bold cyan]")
    logger.info(f"loguru: item {i}")
    time.sleep(0.2)
    return i * 2

def worker_that_fails(i: int) -> int:
    if i % 2 == 1:
        raise ValueError(f"Item {i} is odd")
    return i * 2

print(f"is_notebook: {is_notebook()}")

## Test 1: Auto backend (should use tqdm in notebook)

In [ ]:
results = pmap(worker, ITEMS, n_jobs=2)
assert results == EXPECTED, f"Expected {EXPECTED}, got {results}"
print(f"Results: {results} — PASSED")

## Test 2: Processes + Job Bars (falls back to simple bar in notebook)

In [ ]:
results = pmap(worker, ITEMS, n_jobs=2, show_job_bars=True)
assert results == EXPECTED, f"Expected {EXPECTED}, got {results}"
print(f"Results: {results} — PASSED")

## Test 3: Threads + Simple Bar

In [ ]:
results = pmap(worker, ITEMS, n_jobs=2, prefer='threads')
assert results == EXPECTED, f"Expected {EXPECTED}, got {results}"
print(f"Results: {results} — PASSED")

## Test 4: Threads + Job Bars

In [ ]:
results = pmap(worker, ITEMS, n_jobs=2, prefer='threads', show_job_bars=True)
assert results == EXPECTED, f"Expected {EXPECTED}, got {results}"
print(f"Results: {results} — PASSED")

## Test 5: Sequential (n_jobs=1)

In [ ]:
results = pmap(worker, ITEMS, n_jobs=1)
assert results == EXPECTED, f"Expected {EXPECTED}, got {results}"
print(f"Results: {results} — PASSED")

## Test 6: Progress Disabled

In [ ]:
results = pmap(worker, ITEMS, n_jobs=2, disable_tqdm=True)
assert results == EXPECTED, f"Expected {EXPECTED}, got {results}"
print(f"Results: {results} — PASSED")

## Test 7: safe_mode

In [ ]:
results = pmap(worker_that_fails, range(4), n_jobs=2, safe_mode=True)
assert results[0] == 0
assert isinstance(results[1], dict) and results[1]['error_type'] == 'ValueError'
assert results[2] == 4
assert isinstance(results[3], dict) and results[3]['error_type'] == 'ValueError'
print(f"Results: {results} — PASSED")

## Test 8: Explicit backend='tqdm' override

In [ ]:
results = pmap(worker, ITEMS, n_jobs=2, backend='tqdm')
assert results == EXPECTED, f"Expected {EXPECTED}, got {results}"
print(f"Results: {results} — PASSED")

## All Tests Complete

In [ ]:
print("=" * 60)
print("ALL NOTEBOOK TESTS PASSED!")
print("=" * 60)